# Beyond Ratings: Course-Aware Instructor Effectiveness with Reliability and Tier Stability

**Code-only notebook • provide your own authorized input CSV • seed 42**

This notebook estimates relative instructor support needs from batch-level observations. It is not a causal measure of teaching quality and must not determine appraisal, compensation, or dismissal.

**AI assistance disclosure:** this project was prepared with AI assistance. The supplied assessment prohibits LLM assistance; use as a learning reference unless the evaluator explicitly permits it. No LLM, API, external dataset, AutoML, or external service is called by the analysis code.

## 1. Questions and experimental contract
1. What do outcomes, engagement and feedback jointly suggest?
2. How much do course mix and limited batch evidence affect the answer?
3. Can engagement-only features predict a separate, outcome-only tier for unseen instructors?
4. How stable are descriptive tiers under reasonable policy changes?

We keep two tasks distinct: **descriptive full-composite scoring** and **outcome-only tier prediction**. Predicting full-composite tiers from their ingredients is included only as a clearly labeled circular comparison. No future prediction is claimed because timestamps and learner histories are absent.

**Public sharing status:** stored cell outputs and recorded dataset-specific findings have been removed from this copy. The input dataset, trained model artifacts and generated result files are omitted pending explicit approval for public disclosure. Run locally with an authorized CSV to reproduce the analysis.

## 2. Reproducibility and input
Run all cells in order. Put the original CSV beside the notebook or at `data/instructor_batches.csv`. In Colab, upload the supplied CSV using the Files pane. All analysis definitions are embedded below: no repository import or network request is required. The only analysis dependencies are pandas, numpy, scikit-learn and matplotlib. Standard-library modules handle files and model serialization.

Outputs are recreated under `outputs/` and `models/`. The original CSV is read without modification.

In [ ]:
"""Reproducible, course-aware instructor analysis; no API or LLM calls."""
from pathlib import Path
import json
import pickle
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import f1_score, balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.inspection import permutation_importance

SEED = 42
LEVELS = ['Low', 'Medium', 'High']
OUTCOMES = ['completion_rate', 'avg_score_improvement', 'avg_quiz_score']
ENGAGEMENT = ['avg_watch_time', 'assignment_submission_rate', 'forum_activity_rate']
FEEDBACK = ['avg_feedback_score']
METRICS = OUTCOMES + ENGAGEMENT + FEEDBACK
FEATURES = [f'{m}_{s}' for m in ENGAGEMENT for s in ['mean', 'std']]
FEATURES += ['n_batches', 'n_courses']


def validate(df):
    """Fail explicitly on invalid records rather than silently deleting them."""
    required = ['batch_id', 'instructor_id', 'course_id'] + METRICS + ['dropout_rate', 'feedback_response_rate']
    missing = sorted(set(required) - set(df.columns))
    if missing:
        raise ValueError(f'Missing required columns: {missing}')
    if df[required].isna().any().any():
        raise ValueError('Missing data found: review before training/scoring.')
    if df.batch_id.duplicated().any():
        raise ValueError('Batch IDs must be unique.')
    for m in ENGAGEMENT + ['completion_rate', 'dropout_rate', 'feedback_response_rate']:
        if not df[m].between(0, 1).all():
            raise ValueError(f'Out-of-range rate: {m}')
    if not df.avg_feedback_score.between(1, 5).all():
        raise ValueError('Feedback outside 1–5.')
    # Assessment scales are not documented: do not invent 0–100 bounds.
    if not np.isfinite(df[METRICS + ['dropout_rate', 'feedback_response_rate']].to_numpy()).all():
        raise ValueError('Non-finite numeric measurements.')


def fit_reference(df, adjusted=True):
    """Training-only course baselines, residual scale and fixed score origin."""
    center = df[METRICS].mean()
    course = df.groupby('course_id')[METRICS].mean()
    residual = df[METRICS] - center
    if adjusted:
        baseline = course.reindex(df.course_id).set_axis(df.index)
        residual = df[METRICS] - baseline
    scale = residual.std(ddof=0).replace(0, 1)
    return {'center': center, 'course': course, 'scale': scale, 'adjusted': adjusted,
            'origin': 50.0, 'score_unit': 10.0}


def transform_batches(df, ref):
    """Unseen courses use training global means; held-out data never fit baselines."""
    baseline = pd.DataFrame(np.tile(ref['center'].to_numpy(), (len(df), 1)), index=df.index, columns=METRICS)
    if ref['adjusted']:
        baseline = ref['course'].reindex(df.course_id).set_axis(df.index).fillna(ref['center'])
    z = (df[METRICS] - baseline) / ref['scale']
    z['instructor_id'] = df.instructor_id.to_numpy()
    z['course_id'] = df.course_id.to_numpy()
    return z


def aggregate(df, ref):
    """Equal batch weights: learner counts are unavailable. SD preserves consistency."""
    z = transform_batches(df, ref)
    result = z.groupby('instructor_id')[METRICS].agg(['mean', 'std'])
    result.columns = ['_'.join(c) for c in result.columns]
    result = result.fillna(0)  # Single-batch SD undefined; count preserves that distinction.
    result['n_batches'] = z.groupby('instructor_id').size()
    result['n_courses'] = z.groupby('instructor_id').course_id.nunique()
    return result


def score(a, ref, weights=(.7, .2, .1), k=5):
    """A relative index, not a probability or a validated 0–100 teaching rating."""
    outcome = .45*a.completion_rate_mean + .40*a.avg_score_improvement_mean + .15*a.avg_quiz_score_mean
    engagement = a[[f'{m}_mean' for m in ENGAGEMENT]].mean(axis=1)
    feedback = a.avg_feedback_score_mean
    composite = weights[0]*outcome + weights[1]*engagement + weights[2]*feedback
    reliability = a.n_batches / (a.n_batches+k)
    return ref['origin'] + ref['score_unit']*reliability*composite


def fit_policy(df, adjusted=True, weights=(1., 0., 0.), k=5):
    ref = fit_reference(df, adjusted)
    a = aggregate(df, ref)
    s = score(a, ref, weights, k)
    cuts = np.quantile(s, [1/3, 2/3])
    if cuts[0] >= cuts[1]:
        raise ValueError('Scores cannot support three tiers.')
    return {'reference': ref, 'weights': weights, 'k': k, 'cuts': cuts}


def apply_policy(df, policy):
    a = aggregate(df, policy['reference'])
    s = score(a, policy['reference'], policy['weights'], policy['k'])
    y = pd.Series(np.array(LEVELS)[np.digitize(s, policy['cuts'])], index=a.index, name='tier')
    return a, s, y


def candidates():
    return {
        'Dummy': make_pipeline(SimpleImputer(), DummyClassifier(strategy='prior')),
        'Logistic': make_pipeline(SimpleImputer(), StandardScaler(), LogisticRegression(C=1., max_iter=2000, class_weight='balanced', random_state=SEED)),
        'RandomForest': make_pipeline(SimpleImputer(), RandomForestClassifier(n_estimators=200, max_depth=4, min_samples_leaf=5, class_weight='balanced', random_state=SEED, n_jobs=1))
    }


def evaluate(y, pred):
    return {'macro_f1': float(f1_score(y, pred, labels=LEVELS, average='macro', zero_division=0)),
            'balanced_accuracy': float(balanced_accuracy_score(y, pred))}


def train_experiment(df, output_dir='outputs', model_dir='models'):
    """Select on inner folds; evaluate once on unseen instructors. No chronology implied."""
    out, models = Path(output_dir), Path(model_dir)
    out.mkdir(parents=True, exist_ok=True); models.mkdir(parents=True, exist_ok=True)
    validate(df)
    ids = np.sort(df.instructor_id.unique())
    train_ids, test_ids = train_test_split(ids, test_size=.25, random_state=SEED)
    train = df[df.instructor_id.isin(train_ids)]
    test = df[df.instructor_id.isin(test_ids)]
    rows = []
    # Each fold rebuilds course baselines, scales, shrinkage labels and boundaries.
    for fold, (fit_idx, val_idx) in enumerate(KFold(3, shuffle=True, random_state=SEED).split(train_ids), 1):
        fit = train[train.instructor_id.isin(train_ids[fit_idx])]
        val = train[train.instructor_id.isin(train_ids[val_idx])]
        p = fit_policy(fit)
        a, _, y = apply_policy(fit, p); b, _, v = apply_policy(val, p)
        for name, estimator in candidates().items():
            estimator.fit(a[FEATURES], y)
            rows.append({'fold': fold, 'model': name, **evaluate(v, estimator.predict(b[FEATURES]))})
    cv = pd.DataFrame(rows)
    summary = cv.groupby('model')[['macro_f1', 'balanced_accuracy']].agg(['mean', 'std'])
    winner = cv.groupby('model').macro_f1.mean().idxmax()  # Dummy may legitimately win.
    policy = fit_policy(train)
    a, s, y = apply_policy(train, policy); b, t, v = apply_policy(test, policy)
    selected = candidates()[winner].fit(a[FEATURES], y)
    pred = selected.predict(b[FEATURES])
    result = {'selected_model': winner, 'train_instructors': len(a), 'test_instructors': len(b),
              'test': evaluate(v, pred), 'target': 'course-adjusted outcome-only policy tier',
              'interpretation': 'Concurrent association with a constructed proxy; not future or causal effectiveness.'}
    report = pd.DataFrame(classification_report(v, pred, labels=LEVELS, output_dict=True, zero_division=0)).T
    predictions = pd.DataFrame({'actual_tier': v, 'predicted_tier': pred, 'outcome_score': t})
    prob = selected.predict_proba(b[FEATURES])
    for j, level in enumerate(selected.classes_):
        predictions[f'probability_{level}'] = prob[:, j]
    predictions['max_probability'] = prob.max(axis=1)
    predictions['review_flag'] = (predictions.max_probability < .6) | (b.n_batches < 5)
    # A review heuristic, NOT calibrated uncertainty or a validated rejection rule.
    importance = permutation_importance(selected, b[FEATURES], v, scoring='f1_macro', n_repeats=20, random_state=SEED)
    imp = pd.DataFrame({'feature': FEATURES, 'importance': importance.importances_mean, 'std': importance.importances_std}).sort_values('importance', ascending=False)
    # Explicitly circular auxiliary task: reconstruct full composite tiers.
    full = fit_policy(train, weights=(.7, .2, .1))
    fa, _, fy = apply_policy(train, full); fb, _, fv = apply_policy(test, full)
    full_features = [f'{m}_mean' for m in METRICS] + ['n_batches']
    reconstruct = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(fa[full_features], fy)
    result['circular_reconstruction_test'] = evaluate(fv, reconstruct.predict(fb[full_features]))
    result['circular_warning'] = 'These inputs define their own target. This result is formula reproduction only.'
    cv.to_csv(out/'cv_folds.csv', index=False); summary.to_csv(out/'cv_summary.csv')
    report.to_csv(out/'classification_report.csv'); predictions.to_csv(out/'heldout_predictions.csv')
    imp.to_csv(out/'permutation_importance.csv', index=False)
    pd.DataFrame(confusion_matrix(v, pred, labels=LEVELS), index=LEVELS, columns=LEVELS).to_csv(out/'confusion_matrix.csv')
    (out/'metrics.json').write_text(json.dumps(result, indent=2))
    pd.DataFrame({'instructor_id': ids, 'split': ['train' if i in train_ids else 'test' for i in ids]}).to_csv(out/'split_manifest.csv', index=False)
    with (models/'evaluated_model.pkl').open('wb') as f:
        pickle.dump({'model': selected, 'policy': policy, 'features': FEATURES, 'selected_name': winner}, f)
    # Refit for research inference on all supplied instructors; does not change holdout metrics.
    final_policy = fit_policy(df)
    all_a, all_s, all_y = apply_policy(df, final_policy)
    final = candidates()[winner].fit(all_a[FEATURES], all_y)
    with (models/'instructor_model.pkl').open('wb') as f:
        pickle.dump({'model': final, 'policy': final_policy, 'features': FEATURES, 'selected_name': winner}, f)
    return result, cv, predictions, imp


def predict_batches(df, artifact='models/instructor_model.pkl'):
    """Research inference: input is batch rows, output one prediction per instructor."""
    validate(df)
    with open(artifact, 'rb') as f:
        bundle = pickle.load(f)  # Load only your own trusted artifact.
    a, _, _ = apply_policy(df, bundle['policy'])
    model = bundle['model']; p = model.predict_proba(a[bundle['features']])
    result = pd.DataFrame({'predicted_tier': model.predict(a[bundle['features']]), 'n_batches': a.n_batches}, index=a.index)
    for j, level in enumerate(model.classes_):
        result[f'probability_{level}'] = p[:, j]
    result['review_flag'] = (p.max(axis=1) < .6) | (a.n_batches.to_numpy() < 5)
    known = bundle['policy']['reference']['course'].index
    unseen = df.assign(unseen=~df.course_id.isin(known)).groupby('instructor_id').unseen.mean()
    result['unseen_course_fraction'] = unseen
    result['review_flag'] |= unseen > 0
    return result


In [ ]:
import matplotlib.pyplot as plt
import platform
import sklearn

paths = [Path('data/instructor_batches.csv'), Path('instructor_batches.csv')]
paths += sorted(Path('.').glob('*instructor*csv'))
paths += sorted(Path('/content').glob('*instructor*csv')) if Path('/content').exists() else []
DATA_PATH = next((p for p in paths if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Upload the supplied instructor CSV beside this notebook or under data/.')
df = pd.read_csv(DATA_PATH)
validate(df)
Path('outputs').mkdir(exist_ok=True)
Path('models').mkdir(exist_ok=True)
print('Source:', DATA_PATH)
print('Python', platform.python_version(), '| sklearn', sklearn.__version__)
print('Shape:', df.shape, '| instructors:', df.instructor_id.nunique(), '| courses:', df.course_id.nunique())
print(df.head().to_string(index=False))


## 3. Exploratory analysis and integrity
We inspect missingness, duplicates, documented rate/rating ranges, clipping at extrema, the number of batches per instructor, course coverage, distributions and correlations. Improvement and quiz assessment scales are not documented, so observed ranges are reported without inventing universal bounds. Exploratory results below describe the complete supplied dataset; modeling choices and hyperparameters are fixed rather than optimized against the final test set.

In [ ]:
numeric = df.select_dtypes('number').columns.tolist()
audit = pd.DataFrame({'missing': df.isna().sum(), 'unique': df.nunique()})
print(audit.to_string())
print('Duplicate batch IDs:', df.batch_id.duplicated().sum())
print(df[numeric].describe().round(3).to_string())
counts = df.groupby('instructor_id').size()
print('Batches/instructor:', counts.describe().round(2).to_dict())
print('Completion/dropout correlation:', round(df.completion_rate.corr(df.dropout_rate), 3))
print('Mean absolute departure of completion + dropout from 1:', round((df.completion_rate+df.dropout_rate-1).abs().mean(), 4))
print('Feedback response / feedback rating correlation:', round(df.feedback_response_rate.corr(df.avg_feedback_score), 3))
audit.to_csv('outputs/data_audit.csv')
fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for ax, col in zip(axes.flat, numeric):
    ax.hist(df[col], bins=25, color='#2563eb', alpha=.8)
    ax.set_title(col.replace('_', ' '), fontsize=9)
fig.suptitle('Observed batch-level distributions')
fig.tight_layout()
fig.savefig('outputs/distributions.png', dpi=140)
plt.show()


In [ ]:
corr = df[numeric].corr()
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric)), numeric, rotation=90, fontsize=8)
ax.set_yticks(range(len(numeric)), numeric, fontsize=8)
fig.colorbar(im, ax=ax, label='Pearson correlation')
fig.tight_layout(); fig.savefig('outputs/correlations.png', dpi=140)
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(counts, bins=15, color='#0d9488')
axes[0].set(xlabel='Batches per instructor', ylabel='Instructor count')
df.groupby('course_id').completion_rate.mean().sort_values().plot.bar(ax=axes[1], color='#7c3aed')
axes[1].set(ylabel='Mean completion', title='Observed course mix; not causal difficulty')
fig.tight_layout(); fig.savefig('outputs/coverage.png', dpi=140)
plt.show()
print('Largest absolute off-diagonal correlations:')
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack()
print(upper.reindex(upper.abs().sort_values(ascending=False).index).head(5).round(3).to_string())


**Interpretation guidance.** Inspect the generated integrity and coverage statistics after a local run. Completion and dropout overlap conceptually, so dropout is excluded from scoring to avoid double counting. Course differences can reflect learner selection and instructor assignment. Feedback response rate is descriptive context rather than a teaching-quality reward.

## 4. Transparent score and aggregation
For each metric, subtract its training-course mean and divide by its training residual standard deviation. This standardizes different units. Unseen courses use training global means. Course means are unregularized and may be noisy; sparse instructor–course overlap prevents clean separation of instructor and course effects.

For each instructor, calculate means and standard deviations across batches, plus batch and distinct-course counts. Batch means receive equal weights because learner counts are unavailable. Standard deviations summarize consistency; a one-batch SD is filled with zero and distinguished by its batch count.

The **outcome dimension** is 45% completion, 40% score improvement, and 15% quiz score. Improvement receives substantial weight because it is closer to learning gain; quiz scores receive less because entry ability can confound them. These are policy assumptions, not learned truth. The engagement dimension is the mean of watch time, assignment submission and forum activity. The feedback dimension is the mean rating.

The descriptive **full composite** uses 70% outcomes, 20% engagement, 10% feedback. Shrink its standardized composite toward zero with `n_batches / (n_batches + 5)`, then express it as `50 + 10 × shrunk composite`. This is a relative index, **not a bounded 0–100 scale**. The origin 50 is a policy reference, not an empirical Bayes posterior. `k=5` is a transparent heuristic, not an estimated prior.

Tier boundaries are the training instructors’ score tertiles. High/Medium/Low are relative categories. Held-out classes need not be balanced. For descriptive sensitivity only, all supplied instructors form the reference population.

In [ ]:
descriptive_policy = fit_policy(df, weights=(.7, .2, .1))
a_all, descriptive_scores, descriptive_tiers = apply_policy(df, descriptive_policy)
descriptive = pd.DataFrame({'full_score': descriptive_scores, 'tier': descriptive_tiers,
                            'n_batches': a_all.n_batches, 'n_courses': a_all.n_courses})
print(descriptive.head(10).round(3).to_string())
print('Descriptive tier counts:', descriptive_tiers.value_counts().to_dict())
print('Descriptive cuts:', descriptive_policy['cuts'].round(3))
descriptive.to_csv('outputs/descriptive_instructor_scores.csv')


## 5. Prediction design: prevent circular claims
**Primary target:** course-adjusted, shrunk outcome-only tier. **Predictors:** engagement means and standard deviations, batch count and course count. IDs, feedback, completion, dropout, improvement and quiz scores are excluded. Sharing course reference estimates and batch-count shrinkage still connects target construction with observed data; this is an exploratory concurrent association task, not an independent ground-truth test.

Split 120 instructors into 90 development and 30 test instructors before fitting any scoring reference. A random split uses no assumed chronology. Three-fold development validation rebuilds the course means, scales, labels and tier cuts in every fold. Select the highest mean macro F1 among a dummy classifier, balanced logistic regression and a shallow random forest. The dummy is allowed to win: limited signal is a valid conclusion. There is no wide hyperparameter search and the held-out set does not select the model.

Macro F1 values each tier equally; balanced accuracy averages per-tier recall. The confusion matrix and per-tier precision/recall show whether Low instructors are missed or overflagged. A support team might prioritize Low recall, but false Low labels can unfairly stigmatize instructors. No threshold is optimized without product costs and validation data. Class weighting helps training imbalance; SMOTE is unnecessary for this small, roughly balanced constructed target.

In [ ]:
result, cv, heldout, importance = train_experiment(df)
print(json.dumps(result, indent=2))
print('\nDevelopment CV (mean/std across three folds):')
print(cv.groupby('model')[['macro_f1', 'balanced_accuracy']].agg(['mean', 'std']).round(3).to_string())
print('\nHeld-out per-tier report:')
print(pd.read_csv('outputs/classification_report.csv', index_col=0).round(3).to_string())
print('Held-out target class counts:', heldout.actual_tier.value_counts().to_dict())
print('Review flags:', int(heldout.review_flag.sum()), '/', len(heldout))


In [ ]:
cm = pd.read_csv('outputs/confusion_matrix.csv', index_col=0)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(cm, cmap='Blues')
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, int(cm.iloc[i,j]), ha='center', va='center')
axes[0].set(xticks=range(3), xticklabels=LEVELS, yticks=range(3), yticklabels=LEVELS,
            xlabel='Predicted', ylabel='Actual proxy tier', title='Unseen-instructor test confusion matrix')
stats = cv.groupby('model').macro_f1.agg(['mean', 'std'])
axes[1].bar(stats.index, stats['mean'], yerr=stats['std'].fillna(0), color='#0d9488', capsize=4)
axes[1].set(ylabel='Macro F1', ylim=(0,1), title='Development CV; error bars = fold SD')
fig.tight_layout(); fig.savefig('outputs/evaluation.png', dpi=140)
plt.show()


## 6. Interpretation and circular comparison
Permutation importance measures the loss in held-out macro F1 after shuffling a predictor. Positive values suggest predictive usefulness for this constructed target, zero suggests little evidence, and negative values can occur from sampling noise. Correlated predictors can mask each other; 30 test instructors make these estimates fragile. This post-evaluation diagnostic does not drive model selection.

The auxiliary experiment gives logistic regression the ingredients of the full-composite score. Its measured performance above is explicitly **formula reconstruction**, not evidence of real-world instructor effectiveness. No causal conclusion follows from either experiment.

In [ ]:
print(importance.round(4).to_string(index=False))
fig, ax = plt.subplots(figsize=(9, 5))
ordered = importance.sort_values('importance')
ax.barh(ordered.feature, ordered.importance, xerr=ordered['std'], color='#7c3aed', alpha=.8)
ax.axvline(0, color='black', linewidth=.7)
ax.set(xlabel='Change in test macro F1 (mean ± repeat SD)', title='Permutation importance — exploratory')
fig.tight_layout(); fig.savefig('outputs/importance.png', dpi=140)
plt.show()
positive = importance[importance.importance > 0]
if result['selected_model'] == 'Dummy':
    print('The baseline won development validation. There is insufficient evidence to prefer engagement-based classification here.')
elif len(positive):
    print('Largest positive permutation contributions:', ', '.join(positive.feature.head(3)))
    print('These are associations with the proxy outcome tier, not causal teaching drivers.')
else:
    print('No predictor has a positive measured permutation contribution on this small holdout.')


## 7. Tier stability under alternative policies
Compare full-composite labels using more outcome emphasis, more engagement emphasis, no feedback, raw rather than course-adjusted metrics, shrinkage strengths 0 and 15, and nearby percentile boundaries. Each scenario defines its own population tertiles; agreement measures robustness to the whole alternative policy, not an independent accuracy metric. These descriptive analyses use all data and are never used to improve held-out metrics.

In [ ]:
scenarios = {
    'base': (True, (.7,.2,.1), 5),
    'outcomes_85pct': (True, (.85,.1,.05), 5),
    'engagement_35pct': (True, (.55,.35,.1), 5),
    'no_feedback': (True, (.8,.2,0.), 5),
    'raw_metrics': (False, (.7,.2,.1), 5),
    'no_shrinkage': (True, (.7,.2,.1), 0),
    'stronger_shrinkage': (True, (.7,.2,.1), 15),
}
stability = pd.DataFrame(index=a_all.index)
for name, (adjusted, weights, k) in scenarios.items():
    p = fit_policy(df, adjusted=adjusted, weights=weights, k=k)
    _, _, tiers = apply_policy(df, p)
    stability[name] = tiers
for name, quantiles in [('cuts_30_70', [.3,.7]), ('cuts_35_65', [.35,.65])]:
    stability[name] = np.array(LEVELS)[np.digitize(descriptive_scores, np.quantile(descriptive_scores, quantiles))]
agreement = stability.eq(stability.base, axis=0).mean()
stability['agreement_fraction'] = stability.eq(stability.base, axis=0).mean(axis=1)
stability['distinct_tiers'] = stability[list(scenarios)+['cuts_30_70','cuts_35_65']].nunique(axis=1)
stability.to_csv('outputs/tier_stability.csv')
print('Agreement with base by scenario:')
print(agreement.round(3).to_string())
print('Instructors changing tier under any scenario:', int((stability.distinct_tiers > 1).sum()))
fig, ax = plt.subplots(figsize=(9,4))
agreement.drop('base').plot.bar(ax=ax, color='#2563eb')
ax.set(ylabel='Fraction matching base tier', ylim=(0,1), title='Sensitivity to policy assumptions')
fig.tight_layout(); fig.savefig('outputs/stability.png', dpi=140)
plt.show()


## 8. Batch-resampling uncertainty
Resample each instructor’s observed batches 200 times with replacement, holding course reference values, tier boundaries and batch count fixed. The central 90% interval is a conditional batch-resampling interval, not a calibrated confidence guarantee for true teaching ability. It ignores learner-level sampling, uncertainty in course means, selection bias, time dependence and correlations across batches. A single-batch instructor would have an artificially zero-width interval; flag sparse evidence separately.

In [ ]:
rng = np.random.default_rng(SEED)
ref = descriptive_policy['reference']
z = transform_batches(df, ref)
z['batch_composite'] = (.7*(.45*z.completion_rate + .40*z.avg_score_improvement + .15*z.avg_quiz_score)
                        + .2*z[ENGAGEMENT].mean(axis=1) + .1*z.avg_feedback_score)
intervals = []
for instructor, group in z.groupby('instructor_id'):
    values = group.batch_composite.to_numpy()
    n = len(values)
    samples = rng.choice(values, size=(200,n), replace=True).mean(axis=1)
    samples = 50 + 10*n/(n+5)*samples
    low, high = np.quantile(samples, [.05,.95])
    interval_tiers = np.digitize([low,high], descriptive_policy['cuts'])
    intervals.append({'instructor_id':instructor, 'score':descriptive_scores[instructor],
                      'p05':low, 'p95':high, 'n_batches':n,
                      'crosses_tier_boundary':bool(interval_tiers[0] != interval_tiers[1]),
                      'sparse_evidence':bool(n < 5)})
intervals = pd.DataFrame(intervals).set_index('instructor_id')
intervals.to_csv('outputs/conditional_score_intervals.csv')
print('Intervals crossing a tier boundary:', int(intervals.crosses_tier_boundary.sum()), '/', len(intervals))
print('Sparse evidence:', int(intervals.sparse_evidence.sum()))
plot_data = intervals.sort_values('score').iloc[::6]
fig, ax = plt.subplots(figsize=(10,5))
ax.errorbar(range(len(plot_data)), plot_data.score,
            yerr=[np.maximum(plot_data.score-plot_data.p05,0), np.maximum(plot_data.p95-plot_data.score,0)],
            fmt='o', color='#0d9488', capsize=3)
for cut in descriptive_policy['cuts']:
    ax.axhline(cut, color='#dc2626', linestyle='--', alpha=.7)
ax.set_xticks(range(len(plot_data)), plot_data.index, rotation=90)
ax.set(ylabel='Relative full-composite score', title='Every sixth instructor by score; conditional 90% intervals')
fig.tight_layout(); fig.savefig('outputs/uncertainty.png', dpi=140)
plt.show()


## 9. Mandatory analysis questions

### Which features most influenced instructor effectiveness, and why?
For the **defined score**, outcomes dominate by design (70% of the descriptive composite), especially completion and improvement. This is a value choice, not a discovered result. For the **predictive model**, use the measured permutation-importance table and generated interpretation above. A baseline win or small/negative importances means there is insufficient evidence of useful engagement signal. Do not invent a strong driver to make the project sound successful.

### Which variables could be misleading or confounded?
Quiz results reflect prior ability and assessment difficulty; completion/dropout reflect learner schedules, access and motivation; watch time may reflect confusion or playback behavior; forum participation depends on course format; feedback suffers from selection and popularity bias. The response fraction does not reveal whether respondents are representative. Course adjustment can remove genuine instructor signal or encode assignment patterns. More batches measure exposure and may reflect seniority, not quality. No causal attribution is justified.

### How could this model fail in real-world usage?
Only 120 instructor observations constrain generalization. Constructed labels inherit arbitrary weights and thresholds. Drift in curricula, learners and rating behavior can change all associations. Sparse/unseen courses make baselines unstable. Gaming engagement metrics could increase scores without improving learning. Equal batch weighting ignores cohort size, and averaging hides underserved learner groups. Model probabilities are uncalibrated; the 0.6 review rule is illustrative. Splitting instructors tests a new-instructor setting, not future performance of existing instructors. Training on a descriptive proxy does not create an independent measure of teaching quality.

### What additional data would improve the analysis?
Batch and learner counts; pre-assessment ability; comparable post-assessments; course difficulty and format; timestamps; course version; modality; instructor experience and co-teaching roles; learner attendance and access constraints; item-level feedback and response counts; reasons for dropout; qualitative observation and independently moderated teaching rubrics. Appropriate, consented subgroup data could support fairness auditing with safeguards. Timestamps enable prospective evaluation; independent rubrics enable meaningful target validation.

### Should this model be used for instructor performance evaluation?
**Not as a standalone performance rating.** It could inform optional coaching and contextual human review after independent validation. It should not determine employment, compensation or termination. Instructors need explanation, contextual correction and an appeal path. Validate subgroup performance, course coverage, label stability, calibration and prospective usefulness before any operational use. Human oversight cannot compensate for an invalid target.


## 10. Model artifacts and inference
`evaluated_model.pkl` preserves the model and policy used for the held-out experiment. `instructor_model.pkl` refits the selected model and policy on all 120 instructors for research inference; its performance is **not** independently measured. It includes the training course baselines, scales, shrinkage setting, tier cuts, feature names and estimator.

The engagement-only classifier requires concurrent batch observations. It cannot score a new instructor with no batches. Current input validation requires the complete batch schema even though the classifier uses only engagement/count features. A future product should separate engagement-only input validation from outcome-scoring validation. Probabilities and review flags are advisory. Load pickle files only from a trusted source, in the recorded software environment.

In [ ]:
example_ids = sorted(df.instructor_id.unique())[:3]
example = df[df.instructor_id.isin(example_ids)]
example_predictions = predict_batches(example)
example_predictions.to_csv('outputs/inference_example.csv')
print(example_predictions.round(3).to_string())
assert len(example_predictions) == len(example_ids)
assert example_predictions.predicted_tier.isin(LEVELS).all()
# Verify evaluated artifact reproduces the held-out predictions exactly.
with open('models/evaluated_model.pkl', 'rb') as f:
    saved = pickle.load(f)
manifest = pd.read_csv('outputs/split_manifest.csv')
test_ids = manifest.loc[manifest['split'].eq('test'), 'instructor_id']
test_df = df[df.instructor_id.isin(test_ids)]
a_test, _, _ = apply_policy(test_df, saved['policy'])
assert np.array_equal(saved['model'].predict(a_test[saved['features']]), heldout.predicted_tier.to_numpy())
assert not set(manifest.loc[manifest['split'].eq('train'),'instructor_id']) & set(test_ids)
assert np.isfinite(a_test[saved['features']].to_numpy()).all()
print('Checks passed: split separation, finite features, serialized-model round trip, inference schema.')


## 11. Product recommendation and next experiment
Present the three dimensions, batch count, conditional intervals and stability flags in a coaching review screen. Avoid a leaderboard. For unstable or sparse cases, request context instead of asserting a confident tier. Review course assignments and learner support conditions before interpreting differences.

The next meaningful experiment is prospective: use earlier batches to predict independently validated later learning outcomes, with course and learner context. This dataset lacks the dates, learner-level baseline data and independent effectiveness labels needed for that claim. Honest weak predictive results are more useful than impressive circular accuracy.


In [ ]:
summary_text = f"""# Executed results

Selected by development CV: {result['selected_model']}.
Unseen-instructor test: macro F1 {result['test']['macro_f1']:.3f}; balanced accuracy {result['test']['balanced_accuracy']:.3f}.
Development/test instructors: {result['train_instructors']}/{result['test_instructors']}.
Circular full-composite reconstruction macro F1: {result['circular_reconstruction_test']['macro_f1']:.3f}; formula reproduction only.
Descriptive instructors changing tier in any sensitivity scenario: {(stability.distinct_tiers > 1).sum()}/{len(stability)}.
Conditional intervals crossing boundaries: {intervals.crosses_tier_boundary.sum()}/{len(intervals)}.

These results concern a constructed, concurrent proxy, not independent teaching quality or future performance.
"""
Path('outputs/results_summary.md').write_text(summary_text)
print(summary_text)
